In [1]:
import io
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
from google.colab import files

class PlottingMethods:
    """
    A modular utility class to handle granular HTML-wrapped chart generation.
    Designed for flexible embedding in dashboards or reports.
    """

    @staticmethod
    def generate_bar(df, x_col, y_col, title="Bar Chart"):
        """Generates a standard bar chart returned as an HTML string."""
        if df is None or df.empty:
            return "<p>No data available for plotting.</p>"
        fig = px.bar(df, x=x_col, y=y_col, title=title, template="plotly_white")
        return fig.to_html(full_html=False)

    @staticmethod
    def generate_pie(df, names_col, values_col=None, title="Pie Chart"):
        """Generates a pie chart returned as an HTML string."""
        if df is None or df.empty:
            return "<p>No data available for plotting.</p>"
        fig = px.pie(df, names=names_col, values=values_col, title=title, template="plotly_white")
        return fig.to_html(full_html=False)

    @staticmethod
    def generate_histogram(df, col, bins=30, title="Histogram"):
        """Generates a histogram returned as an HTML string."""
        if df is None or df.empty:
            return "<p>No data available for plotting.</p>"
        fig = px.histogram(df, x=col, nbins=bins, title=title, template="plotly_white")
        return fig.to_html(full_html=False)


class DataInspector:
    """
    An automated engine for CSV data ingestion, sanitization, structural analysis,
    advanced imputation, outlier handling, feature engineering, and statistical interactive visualization.
    """

    def __init__(self):
        """Initializes the DataInspector with an empty DataFrame state."""
        self.df = None
        self.numeric_cols = []
        self.categorical_cols = []

    def upload_data(self):
        """
        Handles interactive local file uploads inside Google Colab environment.
        Automatically converts garbage strings into NaN and updates column classification.
        """
        print("Please upload your CSV file:")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded.")
            return None

        filename = list(uploaded.keys())[0]

        # Define typical garbage strings found in real-world messy datasets
        garbage_strings = ['?', 'n/a', 'N/A', 'NaN', 'NULL', 'null', ' ', '']

        # Ingest data, treating garbage strings as NaN
        self.df = pd.read_csv(io.BytesIO(uploaded[filename]), na_values=garbage_strings)
        print(f"\nSuccessfully loaded {filename}!")

        # Auto-type correction & type classification
        self._sanitize_and_classify()
        return self.df

    def _sanitize_and_classify(self):
        """Internal helper to convert numeric columns masquerading as strings and update classifications."""
        if self.df is None:
            return

        for col in self.df.columns:
            # Drop NaNs temporarily to check if the remaining data is purely numeric
            non_null_data = self.df[col].dropna()
            if not non_null_data.empty:
                # Attempt forcing to numeric
                converted = pd.to_numeric(self.df[col], errors='coerce')
                # If the conversion doesn't result in an entirely null column, keep it
                if not converted.dropna().empty and converted.notnull().sum() >= (self.df[col].notnull().sum() * 0.5):
                    self.df[col] = converted

        # Identify numeric and categorical columns
        self.numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

    def display_summary(self):
        """Displays data shape structural dimensions, column breakdown, and a 20-row preview."""
        if self.df is None:
            print("No dataset loaded.")
            return

        print("="*60)
        print(" DATASET SUMMARY ")
        print("="*60)
        print(f"Total Rows: {self.df.shape[0]}")
        print(f"Total Columns: {self.df.shape[1]}")
        print(f"\nNumeric Columns ({len(self.numeric_cols)}): {self.numeric_cols}")
        print(f"Categorical Columns ({len(self.categorical_cols)}): {self.categorical_cols}")
        print("\nMissing Values Per Column:")
        print(self.df.isnull().sum()[self.df.isnull().sum() > 0])
        print("\nFirst 20 Rows Preview:")
        display(self.df.head(20))

    def handle_missing_values(self, strategy='median', fill_value=None):
        """
        Imputes missing values using multiple customizable strategies: 'mean', 'median', 'mode', 'constant'.
        """
        if self.df is None:
            print("No dataset loaded.")
            return

        for col in self.df.columns:
            if self.df[col].isnull().any():
                if strategy == 'mean' and col in self.numeric_cols:
                    self.df[col] = self.df[col].fillna(self.df[col].mean())
                elif strategy == 'median' and col in self.numeric_cols:
                    self.df[col] = self.df[col].fillna(self.df[col].median())
                elif strategy == 'mode' or (strategy in ['mean', 'median'] and col in self.categorical_cols):
                    # Mode can return multiple values, take the first one if available
                    mode_val = self.df[col].mode()
                    if not mode_val.empty:
                        self.df[col] = self.df[col].fillna(mode_val[0])
                elif strategy == 'constant':
                    self.df[col] = self.df[col].fillna(fill_value if fill_value is not None else "Missing")
        print(f"Missing values handled using strategy: '{strategy}'")

    def remove_duplicates(self):
        """Prunes exact row matches from the internal dataset."""
        if self.df is None:
            return
        initial_rows = self.df.shape[0]
        self.df = self.df.drop_duplicates().reset_index(drop=True)
        print(f"Removed {initial_rows - self.df.shape[0]} duplicate rows.")

    def handle_outliers(self, target_columns, action='flag'):
        """
        Uses an IQR-based system to detect outliers in target numeric columns.
        Actions supported: 'flag' (adds a column tracking outliers) or 'delete' (drops rows).
        """
        if self.df is None:
            return

        outlier_mask = pd.Series(False, index=self.df.index)

        for col in target_columns:
            if col in self.numeric_cols:
                Q1 = self.df[col].quantile(0.25)
                Q3 = self.df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR

                col_outliers = (self.df[col] < lower_bound) | (self.df[col] > upper_bound)
                outlier_mask = outlier_mask | col_outliers

        if action == 'delete':
            initial_shape = self.df.shape[0]
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f"Deleted {initial_shape - self.df.shape[0]} rows identified as outliers.")
        else:
            self.df['is_outlier'] = outlier_mask.astype(int)
            print("Outliers flagged under the column 'is_outlier'.")

    def delete_columns(self, input_string):
        """Accepts comma-separated text string to prune specific columns."""
        if self.df is None:
            return
        cols_to_delete = [c.strip() for c in input_string.split(',') if c.strip() in self.df.columns]
        self.df.drop(columns=cols_to_delete, inplace=True)
        self._sanitize_and_classify()
        print(f"Deleted columns: {cols_to_delete}")

    def delete_rows(self, indices_string):
        """Accepts comma-separated text string representing integer indices to prune specific rows."""
        if self.df is None:
            return
        try:
            indices_to_delete = [int(i.strip()) for i in indices_string.split(',') if i.strip().isdigit()]
            valid_indices = [i for i in indices_to_delete if i in self.df.index]
            self.df.drop(index=valid_indices, inplace=True)
            self.df.reset_index(drop=True, inplace=True)
            print(f"Deleted rows with indices: {valid_indices}")
        except Exception as e:
            print(f"Error parsing row indices: {e}")

    def extract_normalized_numeric_data(self, strategy='standard'):
        """
        Normalizes numeric columns.
        Strategies: 'minmax' [0,1], 'standard' (Z-score), 'robust' (IQR scaling via median).
        """
        if self.df is None or not self.numeric_cols:
            return pd.DataFrame()

        scaled_df = pd.DataFrame(index=self.df.index)
        for col in self.numeric_cols:
            vals = self.df[col]
            if strategy == 'minmax':
                min_v, max_v = vals.min(), vals.max()
                scaled_df[f"{col}_minmax"] = (vals - min_v) / (max_v - min_v) if max_v != min_v else 0
            elif strategy == 'robust':
                q1, q2, q3 = vals.quantile(0.25), vals.quantile(0.50), vals.quantile(0.75)
                iqr = q3 - q1
                scaled_df[f"{col}_robust"] = (vals - q2) / iqr if iqr != 0 else 0
            else: # Standard Z-score
                mean, std = vals.mean(), vals.std()
                scaled_df[f"{col}_standard"] = (vals - mean) / std if std != 0 else 0
        return scaled_df

    def extract_normalized_categorical_data(self, strategy='onehot'):
        """
        Encodes categorical elements.
        Strategies: 'onehot' (dummy coding), 'ordinal' (ranked integers), 'uniform' (scaled 0 to 1).
        """
        if self.df is None or not self.categorical_cols:
            return pd.DataFrame()

        if strategy == 'onehot':
            return pd.get_dummies(self.df[self.categorical_cols], drop_first=True, dtype=int)

        encoded_df = pd.DataFrame(index=self.df.index)
        for col in self.categorical_cols:
            # Map elements to specific ordering
            categories = self.df[col].astype('category').cat.categories
            mapping = {cat: idx for idx, cat in enumerate(categories)}
            ordinal_vals = self.df[col].map(mapping)

            if strategy == 'uniform':
                max_val = len(categories) - 1
                encoded_df[f"{col}_uniform"] = ordinal_vals / max_val if max_val > 0 else 0
            else: # Ordinal
                encoded_df[f"{col}_ordinal"] = ordinal_vals
        return encoded_df

    def generate_unified_dataset(self, num_strategy='standard', cat_strategy='onehot'):
        """Merges processed normalized numerical data and encoded categorical components into one unified dataframe."""
        num_df = self.extract_normalized_numeric_data(strategy=num_strategy)
        cat_df = self.extract_normalized_categorical_data(strategy=cat_strategy)
        return pd.concat([num_df, cat_df], axis=1)

    def plot_univariate_subplots(self, column):
        """Generates a high-fidelity 3-panel subplot layout for any single target continuous column."""
        if self.df is None or column not in self.numeric_cols:
            print("Invalid numeric column selected.")
            return

        fig = make_subplots(
            rows=3, cols=1,
            subplot_titles=(f"Violin Plot: {column}", f"Scatter Distribution (Index vs Value)", f"Histogram: {column}"),
            vertical_spacing=0.1
        )

        # Panel 1: Violin / Box Combo
        fig.add_trace(go.Violin(x=self.df[column], box_visible=True, points='all', name=column, marker_color='royalblue'), row=1, col=1)
        # Panel 2: Scatter (Index vs Value)
        fig.add_trace(go.Scatter(x=self.df.index, y=self.df[column], mode='markers', marker=dict(color='teal'), name='Value Index'), row=2, col=1)
        # Panel 3: Histogram
        fig.add_trace(go.Histogram(x=self.df[column], nbinsx=30, marker_color='coral', name='Frequency'), row=3, col=1)

        fig.update_layout(height=800, width=900, title_text=f"Univariate Profiling Engine — {column}", showlegend=False, template="plotly_white")
        fig.show()

    def plot_relationship(self, col1, col2):
        """Dynamically identifies variable types to select the mathematically correct chart format."""
        if self.df is None:
            return

        c1_num = col1 in self.numeric_cols
        c2_num = col2 in self.numeric_cols

        # Case A: Numeric vs Numeric -> Scatter with OLS Trendline
        if c1_num and c2_num:
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"Numeric-Numeric Relationship: {col1} vs {col2}", template="plotly_white")
            fig.show()

        # Case B: Categorical vs Numeric (or vice versa) -> Box plot
        elif (not c1_num and c2_num) or (c1_num and not c2_num):
            cat_c = col1 if not c1_num else col2
            num_c = col2 if c2_num else col1
            fig = px.box(self.df, x=cat_c, y=num_c, points="all", title=f"Categorical-Numeric Relationship: {num_c} by {cat_c}", template="plotly_white")
            fig.show()

        # Case C: Categorical vs Categorical -> Grouped Bar Chart
        else:
            grouped = self.df.groupby([col1, col2]).size().reset_index(name='counts')
            fig = px.bar(grouped, x=col1, y='counts', color=col2, barmode='group', title=f"Categorical-Categorical Relationship: {col1} vs {col2}", template="plotly_white")
            fig.show()

    def plot_categorical_frequency(self, column):
        """Creates a bar chart displaying raw counts with structured percent metrics as explicit labels."""
        if self.df is None or column not in self.categorical_cols:
            print("Invalid categorical column selected.")
            return

        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, 'count']
        total = counts['count'].sum()
        counts['percentage'] = (counts['count'] / total * 100).round(2).astype(str) + '%'

        fig = px.bar(
            counts, x=column, y='count',
            text='percentage',
            title=f"Categorical Frequency Distribution: {column}",
            labels={'count': 'Raw Counts'},
            template="plotly_white"
        )
        fig.update_traces(textposition='outside')
        fig.show()

    def plot_all_associations_heatmap(self):
        """
        Constructs an advanced correlation heatmap across all data types using:
        - Pearson's r (Num-Num)
        - Cramer's V (Cat-Cat)
        - Eta / One-way ANOVA (Num-Cat)
        """
        if self.df is None:
            return

        cols = self.df.columns.tolist()
        n = len(cols)
        matrix = np.zeros((n, n))

        for i in range(n):
            for j in range(n):
                col1, col2 = cols[i], cols[j]

                if i == j:
                    matrix[i][j] = 1.0
                    continue

                c1_num = col1 in self.numeric_cols
                c2_num = col2 in self.numeric_cols

                # 1. Numeric-Numeric: Pearson r
                if c1_num and c2_num:
                    valid_data = self.df[[col1, col2]].dropna()
                    if len(valid_data) > 1:
                        r, _ = stats.pearsonr(valid_data[col1], valid_data[col2])
                        matrix[i][j] = np.nan_to_num(r)

                # 2. Categorical-Categorical: Cramér's V
                elif not c1_num and not c2_num:
                    confusion_matrix = pd.crosstab(self.df[col1], self.df[col2])
                    if not confusion_matrix.empty and confusion_matrix.sum().sum() > 0:
                        chi2 = stats.chi2_contingency(confusion_matrix)[0]
                        n_obs = confusion_matrix.sum().sum()
                        r, c = confusion_matrix.shape
                        if n_obs > 0 and min(r-1, c-1) > 0:
                            matrix[i][j] = np.sqrt(chi2 / (n_obs * min(r - 1, c - 1)))

                # 3. Mixed (Num-Cat): Eta coefficient squared root (via ANOVA)
                else:
                    num_c = col1 if c1_num else col2
                    cat_c = col2 if c1_num else col1

                    valid_data = self.df[[num_c, cat_c]].dropna()
                    groups = [group[num_c].values for name, group in valid_data.groupby(cat_c)]

                    if len(groups) > 1 and sum(len(g) for g in groups) > len(groups):
                        f_val, _ = stats.f_oneway(*groups)
                        # Calculate Eta-squared: SS_between / SS_total
                        # Approximate derivation via F-statistic:
                        k = len(groups)
                        N = sum(len(g) for g in groups)
                        df_between = k - 1
                        df_within = N - k
                        if (f_val * df_between + df_within) != 0:
                            eta_sq = (f_val * df_between) / (f_val * df_between + df_within)
                            matrix[i][j] = np.sqrt(eta_sq)

        # Render Heatmap canvas
        fig = go.Figure(data=go.Heatmap(
            z=matrix,
            x=cols,
            y=cols,
            colorscale='RdBu',
            zmin=-1.0, zmax=1.0,
            text=np.round(matrix, 2),
            texttemplate="%{text}",
            hoverinfo="z"
        ))
        fig.update_layout(title="Unified Association Heatmap (Pearson / Cramér's V / Eta)", template="plotly_white", width=800, height=800)
        fig.show()

In [2]:
# 1. Initialize our engine
inspector = DataInspector()

# 2. Programmatically fetch Titanic dataset directly
import seaborn as sns
print("Downloading real-world Titanic validation set...")
raw_titanic = sns.load_dataset('titanic')

# Inject artificial garbage strings to test the engine
raw_titanic.iloc[2, 2] = '?'
raw_titanic.iloc[5, 3] = 'NULL'
raw_titanic.iloc[10, 4] = ' '

# BYPASS OPTION: Force the data straight into the inspector variable
inspector.df = raw_titanic
inspector._sanitize_and_classify()
print("\n--- STEP 1 & 2: Data Successfully Injected and Sanitized ---")

# 3. Run the rest of the Pipeline Execution automatically
print("\n--- STEP 3: Structural Verification ---")
inspector.display_summary()

print("\n--- STEP 4: Handle Missing Imputations ---")
inspector.handle_missing_values(strategy='median')

print("\n--- STEP 5: Remove Duplicates & Outlier Handling ---")
inspector.remove_duplicates()
inspector.handle_outliers(target_columns=['age', 'fare'], action='flag')

print("\n--- STEP 6: Feature Engineering Preparation & Normalization ---")
unified_df = inspector.generate_unified_dataset(num_strategy='standard', cat_strategy='onehot')
print("Unified Engineered Dimensions Frame Sample Output:")
display(unified_df.head(5))

print("\n--- STEP 7: High Fidelity Data Visualization Analysis ---")
if inspector.numeric_cols:
    inspector.plot_univariate_subplots(column='fare')
if len(inspector.numeric_cols) >= 2:
    inspector.plot_relationship(col1='age', col2='fare')
if inspector.categorical_cols:
    inspector.plot_categorical_frequency(column='class')

print("\n--- STEP 8: Unified Matrix Deep Statistical Analysis ---")
inspector.plot_all_associations_heatmap()

/tmp/ipykernel_3315/4169099046.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NULL' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  raw_titanic.iloc[5, 3] = 'NULL'
/tmp/ipykernel_3315/4169099046.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' ' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  raw_titanic.iloc[10, 4] = ' '



--- STEP 1 & 2: Data Successfully Injected and Sanitized ---

--- STEP 3: Structural Verification ---
 DATASET SUMMARY 
Total Rows: 891
Total Columns: 15

Numeric Columns (6): ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
Categorical Columns (9): ['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']

Missing Values Per Column:
age            177
sibsp            1
embarked         2
deck           688
embark_town      2
dtype: int64

First 20 Rows Preview:


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1.0,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1.0,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,?,26.0,0.0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1.0,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0.0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0.0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
6,0,1,male,54.0,0.0,0,51.8625,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3.0,1,21.0750,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0.0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1.0,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False



--- STEP 4: Handle Missing Imputations ---
Missing values handled using strategy: 'median'

--- STEP 5: Remove Duplicates & Outlier Handling ---
Removed 112 duplicate rows.
Outliers flagged under the column 'is_outlier'.

--- STEP 6: Feature Engineering Preparation & Normalization ---
Unified Engineered Dimensions Frame Sample Output:


,survived_standard,pclass_standard,age_standard,sibsp_standard,parch_standard,fare_standard,adult_male,alone,sex_female,sex_male,...,who_woman,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G,embark_town_Queenstown,embark_town_Southampton,alive_yes
0,-0.838863,0.885165,-0.552190,0.480477,-0.498825,-0.526190,True,False,0,1,...,0,0,1,0,0,0,0,0,1,0
1,1.190560,-1.456239,0.612870,0.480477,-0.498825,0.698077,False,False,1,0,...,1,0,1,0,0,0,0,0,0,1
2,1.190560,0.885165,-0.260925,-0.531122,-0.498825,-0.513285,False,True,0,0,...,1,0,1,0,0,0,0,0,1,1
3,1.190560,-1.456239,0.394421,0.480477,-0.498825,0.350427,False,False,1,0,...,1,0,1,0,0,0,0,0,1,1
4,-0.838863,0.885165,0.394421,-0.531122,-0.498825,-0.510895,True,True,0,1,...,0,0,1,0,0,0,0,0,1,0



--- STEP 7: High Fidelity Data Visualization Analysis ---



--- STEP 8: Unified Matrix Deep Statistical Analysis ---


/tmp/ipykernel_3315/3059134089.py:367: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_3315/3059134089.py:367: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite

/tmp/ipykernel_3315/3059134089.py:367: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence th